# Build Entity Dataset

Demonstrates use of the Intelligence Toolkit library to build a structured dataset of entities from AI-driven web search.

Research runs in a background thread, so the notebook polls `progress` until the run completes. Note that this workflow performs live web searches and incurs API costs: keep `max_queries` and `budget` small while experimenting.

See [readme](https://github.com/microsoft/intelligence-toolkit/blob/main/app/workflows/build_entity_dataset/README.md) for more details.

In [ ]:
import sys

sys.path.append("..")
import os
import time

from intelligence_toolkit.build_entity_dataset.api import BuildEntityDataset

In [ ]:
# Create the workflow object
bed = BuildEntityDataset()

# Describe the kind of entity to research, plus optional guidance to focus the search
category = "Anti-trafficking non-profit organizations in Southeast Asia"
guidance = "Focus on organizations with a public website and an identifiable country of operation."

# Optionally pin the attributes to collect. Omit to let the workflow propose a schema.
schema_attributes = [
    {"name": "country", "description": "Primary country of operation", "is_multi_valued": False},
    {"name": "website", "description": "Official website URL", "is_multi_valued": False},
    {"name": "focus_area", "description": "Main areas of work, e.g. victim support, advocacy, research"},
]

In [ ]:
# Start research in the background
bed.start_research(
    api_key=os.environ["OPENAI_API_KEY"],
    category=category,
    guidance=guidance,
    schema_attributes=schema_attributes,
    max_queries=10,
    budget=1.0,
    verify=False,
)

# Poll until the run finishes
while bed.is_running:
    bed.refresh_progress()
    p = bed.progress
    print(f"{p.stage}: {p.entity_count} entities, {p.query_count} queries")
    time.sleep(10)

if bed.progress.error:
    raise RuntimeError(bed.progress.error)
print(f"Done: {bed.progress.entity_count} entities, ${bed.usage.total_cost_usd:.2f} spent")

In [ ]:
# Inspect the resulting dataset
df = bed.current_dataframe()
df.head()

In [ ]:
# Export the dataset as CSV and JSON
output_dir = "../example_outputs/build_entity_dataset"
os.makedirs(output_dir, exist_ok=True)
with open(f"{output_dir}/entities.csv", "wb") as f:
    f.write(bed.get_dataset_bytes_csv())
with open(f"{output_dir}/entities.json", "wb") as f:
    f.write(bed.get_dataset_bytes_json())
print(f"Saved to {output_dir}; run cached at {bed.run_dir}")